In [1]:
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")          # reads ANTHROPIC_API_KEY from .env into the environment
client = anthropic.Anthropic()  # finds the key automatically; it never appears in your code

response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello in five words."}],
)
print(response.content[0].text)

Hello there, friend. How are you?


In [2]:
SYSTEM = """You answer questions about board and card game rules, using ONLY the rulebook excerpts provided.

Rules:
1. Use only the excerpts. Do not use outside knowledge, even if you know the game well.
2. Each excerpt is labeled with its game. Only use excerpts from the game the question asks about.
   If the question names no game, answer separately for each game whose excerpts actually answer it, labeled by game.
   Leave out games whose excerpts don't address the question.
   If the question uses a term that belongs to only one game (e.g. "hole cards" for Texas Hold'em), treat it as about that game.
3. After each claim, cite the excerpt id in square brackets, e.g. [catan-p9-c0].
4. If the excerpts do not contain the answer, reply exactly: "The provided rules don't cover this."
   You may then mention the closest related rule, clearly labeled as related.
5. Be short and direct."""

In [3]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":   # only move up if we're still in notebooks/
    os.chdir("..")
sys.path.append("src")
print("Working in:", os.getcwd())    # should end in /Game
from rulebook_rag.retrieve import Retriever , detect_games


retriever = Retriever()

def format_context(chunks):
    parts = []
    for c in chunks:
        parts.append(f"<excerpt id=\"{c['id']}\" game=\"{c['game']}\" page=\"{c['page']}\">\n{c['text']}\n</excerpt>")
    return "\n\n".join(parts)

def answer(question, k=5):
    chunks = retriever.search_routed(question)
    user_message = f"Excerpts:\n{format_context(chunks)}\n\nQuestion: {question}"
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=2000,                      # was 400
        system=SYSTEM,
        messages=[{"role": "user", "content": user_message}],
    )
    if response.stop_reason == "max_tokens":
        print("WARNING: response was cut off")
    text = "".join(b.text for b in response.content if b.type == "text")
    return text, chunks

Working in: /Users/yasasvikotagiri/Desktop/Game


/Users/yasasvikotagiri/Desktop/Game/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1027 10:34:39.964000 50242 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9006.73it/s]


In [4]:
for q in [
    "How many private hole cards is each player dealt?",           # direct lookup
    "Do you collect money for landing on Free Parking?",           # myth-buster
    "What penalty does the CATAN rulebook impose for flipping the board?",  # unanswerable
]:
    text, chunks = answer(q)
    print("Q:", q)
    print("A:", text)
    print("Chunks:", [c["id"] for c in chunks])
    print("=" * 60)

Q: How many private hole cards is each player dealt?
A: Each player is dealt **two** private hole cards. [texas_holdem-p2-c1]

The other excerpt says the same: every player is dealt two cards, for their eyes only. [texas_holdem-p2-c0]
Chunks: ['texas_holdem-p2-c1', 'rummy-p4-c1', 'texas_holdem-p2-c0', 'poker-p6-c1', 'poker-p1-c2', 'blackjack-p1-c0', 'blackjack-p2-c0', 'catan-p2-c1', 'catan-p5-c0', 'monopoly-p2-c1', 'monopoly-p1-c5']
Q: Do you collect money for landing on Free Parking?
A: No. Landing on Free Parking gives you nothing: "Relax! Nothing happens." [monopoly-p2-c5] The rules also state that you don't get a bonus for landing on Free Parking. [monopoly-p1-c0]
Chunks: ['monopoly-p1-c0', 'monopoly-p2-c5', 'monopoly-p2-c3', 'monopoly-p2-c0', 'monopoly-p2-c6']
Q: What penalty does the CATAN rulebook impose for flipping the board?
A: The provided rules don't cover this.

Related (not a penalty): The setup places the robber on the desert hex [catan-p11-c0], and the turn begins with 

In [5]:
import csv

with open("eval/questions.csv") as f:
    questions = list(csv.DictReader(f))

results = []
for row in questions:
    text, chunks = answer(row["question"])
    results.append({
        "id": row["id"],
        "question_type": row["question_type"],
        "question": row["question"],
        "expected_answer": row["expected_answer"],
        "answer": text,
        "chunks": ";".join(c["id"] for c in chunks),
    })
    print(row["id"], "done")

with open("eval/results_v6.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print("Saved", len(results), "results")

1 done
2 done
3 done
4 done
5 done
6 done
7 done
8 done
9 done
10 done
11 done
12 done
13 done
14 done
15 done
16 done
17 done
18 done
19 done
20 done
21 done
22 done
23 done
24 done
25 done
26 done
27 done
28 done
29 done
30 done
31 done
32 done
33 done
34 done
35 done
36 done
37 done
38 done
39 done
40 done
41 done
Saved 41 results


In [6]:
for r in results:
    if r["question_type"] in ("unanswerable", "myth_buster"):
        print(f"[{r['id']}] {r['question_type']}")
        print("Q:       ", r["question"])
        print("EXPECTED:", r["expected_answer"])
        print("GOT:     ", r["answer"])
        print("-" * 60)

[3] myth_buster
Q:        Is every development card forbidden from being used on the turn you buy it?
EXPECTED: No. Victory Point cards bought that turn may be revealed if they give you enough points to win.
GOT:      No. Most development cards can't be played on the turn you buy them, but Victory Point cards are an exception.

- Development cards generally: "It may not be a card you built this turn." [catan-p9-c0]
- Victory Point exception: "You may play multiple VP cards (even on the turn you buy them) in order to win the game." [catan-p3-c1]
------------------------------------------------------------
[5] unanswerable
Q:        What penalty does the CATAN rulebook impose for flipping over the board?
EXPECTED: Not specified in the supplied rulebook.
GOT:      The provided rules don't cover this.

Related but not a penalty: the excerpts describe the board setup, such as placing hexes and number discs inside the frame [catan-p11-c0], but say nothing about penalties for disturbing the b

In [7]:
by_id = {c["id"]: c for c in retriever.chunks}
print(by_id["texas_holdem-p2-c2"]["text"])

In Hold'em, a marker called ‘the button' or ‘the dealer button' indicates which player is the dealer for the current game. Before the game begins, the player immediately clockwise from the button posts the "small blind", the first forced bet. The player immediately clockwise from the small blind posts the "big blind", which is typically twice the size of the small blind, but the blinds can vary depending on the stakes and betting structure being played.

In Limit games, the big blind is the same as the small bet, and the small blind is typically half the size of the big blind but may be larger depending on the stakes. For example, in a 4 Limit 2/ game the small blind is $1 and the big blind is $2. In a 30 Limit game, the small blind is $10 15/ and the big blind is $15.


In [8]:
print("1:1" in by_id["blackjack-p2-c0"]["text"])

True


## Faithfulness v2

- Unanswerable: 6/6 correct refusals
- Myth-buster: 6/6 correct, all cited
- Related-rule tails: checked rows 22 and 36 against the chunks — grounded
- New extraction bug: Hold'em "$15/$30 Limit" garbled to "30 Limit ... $10 15/"

In [9]:
JUDGE_SYSTEM = """You grade answers to rulebook questions.
Compare the ANSWER with the EXPECTED answer.
- PASS if the answer states the same facts as EXPECTED (wording may differ; extra correct detail is fine).
- FAIL if it contradicts EXPECTED, misses a key fact, or invents a rule.
- If EXPECTED says the rules don't specify it, PASS only if the answer clearly says the rules don't cover it.
Reply in exactly this format:
VERDICT: PASS or FAIL
REASON: one sentence"""

def judge(question, expected, answer_text):
    msg = f"QUESTION: {question}\nEXPECTED: {expected}\nANSWER: {answer_text}"
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=2000,
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": msg}],
    )

    if response.stop_reason == "max_tokens":
        print("WARNING: response was cut off")
    out = "".join(b.text for b in response.content if b.type == "text")
    if "VERDICT: PASS" in out:
        verdict = "PASS"
    elif "VERDICT: FAIL" in out:
        verdict = "FAIL"
    else:
        verdict = "ERROR"     # the judge didn't answer, so we don't know
    return verdict, out


for r in results:
    r["verdict"], r["judge_note"] = judge(r["question"], r["expected_answer"], r["answer"])
    print(r["id"], r["question_type"], r["verdict"])

1 direct_lookup PASS
2 multi_hop FAIL
3 myth_buster PASS
4 multi_hop PASS
5 unanswerable PASS
6 table_figure FAIL
7 direct_lookup PASS
8 multi_hop FAIL
9 myth_buster PASS
10 direct_lookup PASS
11 unanswerable PASS
12 table_figure FAIL
13 direct_lookup PASS
14 multi_hop FAIL
15 myth_buster PASS
16 disambiguation PASS
17 unanswerable PASS
18 table_figure FAIL
19 direct_lookup PASS
20 direct_lookup PASS
21 multi_hop FAIL
22 myth_buster PASS
23 disambiguation PASS
24 unanswerable PASS
25 direct_lookup PASS
26 direct_lookup PASS
27 multi_hop PASS
28 myth_buster PASS
29 disambiguation PASS
30 unanswerable PASS
31 direct_lookup PASS
32 direct_lookup PASS
33 multi_hop PASS
34 myth_buster PASS
35 disambiguation PASS
36 unanswerable PASS
37 disambiguation PASS
38 disambiguation FAIL
39 disambiguation PASS
40 disambiguation PASS
41 disambiguation FAIL


In [10]:
for r in results:
    if r["verdict"] == "FAIL":
        print(f"[{r['id']}] {r['question_type']}")
        print("Q:       ", r["question"])
        print("EXPECTED:", r["expected_answer"])
        print("GOT:     ", r["answer"])
        print("JUDGE:   ", r["judge_note"])
        print("-" * 60)

[2] multi_hop
Q:        Can I build another settlement on my first turn after setup?
EXPECTED: Yes, during the Action phase, if you can pay its cost, connect it to your road, obey the Distance Rule, and have a settlement available. Setup does not guarantee that you can meet those conditions.
GOT:      Yes, if you meet the settlement requirements. A new settlement must be at least two edges from all other buildings and must connect to one of your existing roads [catan-p8-c2]. Setup gives each player one settlement and road per round, so you'll have roads to build from [catan-p12-c0]. You have 5 settlement pieces, and you must upgrade one to a city before building more than that [catan-p8-c2].

The excerpts don't give the settlement's resource cost, so check that you have the required resources in your hand.
JUDGE:    VERDICT: FAIL
REASON: The answer gets the core "yes, if you meet the requirements" right, but it wrongly says setup gives each player one settlement and road "per round" an

In [11]:
   print([r["id"] for r in results if not r["answer"].strip()])

[]


In [12]:
   from collections import Counter
   total = Counter(r["question_type"] for r in results)
   passed = Counter(r["question_type"] for r in results if r["verdict"] == "PASS")
   for t in total:
       print(f"{t}: {passed[t]}/{total[t]}")

direct_lookup: 10/10
multi_hop: 3/7
myth_buster: 6/6
unanswerable: 6/6
table_figure: 0/3
disambiguation: 7/9


In [13]:
with open("eval/results_v6.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

In [14]:
for r in results:
    if r["question_type"] == "disambiguation" and r["verdict"] == "FAIL":
        print(r["id"], "|", r["question"], "|", r["chunks"])

38 | What is the first forced bet in a hand? | poker-p3-c4;blackjack-p1-c1;texas_holdem-p4-c0;poker-p3-c0;texas_holdem-p2-c0;rummy-p7-c0;rummy-p3-c1
41 | Who goes first? | catan-p12-c0;rummy-p6-c1;texas_holdem-p4-c0;rummy-p7-c0;poker-p6-c0;blackjack-p1-c1;blackjack-p1-c0;monopoly-p2-c1;monopoly-p1-c2


In [15]:
games = sorted({c["game"] for c in retriever.chunks})

def best_score_per_game(question):
    for game in games:
        in_game = [c for c in retriever.dense(question, k=113) if c["game"] == game][:10]
        scores = retriever.reranker.predict([(question, c["text"]) for c in in_game])
        print(f"  {game:13s} best score: {max(scores):6.2f}")

for q in ["What is an Ace worth?", "Who goes first?", "What happens if there's a tie?"]:
    print(q)
    best_score_per_game(q)

What is an Ace worth?
  blackjack     best score:  -5.12
  catan         best score:  -9.80
  monopoly      best score: -11.17
  poker         best score:  -6.60
  rummy         best score:   1.18
  texas_holdem  best score:  -6.63
Who goes first?
  blackjack     best score:  -7.02
  catan         best score:  -2.57
  monopoly      best score:   0.85
  poker         best score:  -3.85
  rummy         best score:  -2.44
  texas_holdem  best score:  -5.12
What happens if there's a tie?
  blackjack     best score:  -7.41
  catan         best score: -10.76
  monopoly      best score: -10.43
  poker         best score:  -3.40
  rummy         best score:  -8.44
  texas_holdem  best score:  -9.40


In [16]:

for q in ["What is an Ace worth?",
          "In Blackjack, is an Ace always worth one point, as it is by default in Standard Rummy?",
          "In Blackjack, at what hand total may the dealer stop taking cards?"]:
    print(q, "→", detect_games(q))
    print("   ", [c["id"] for c in retriever.search_routed(q)])

What is an Ace worth? → []
    ['rummy-p2-c2', 'rummy-p2-c1', 'rummy-p6-c2', 'blackjack-p2-c0', 'blackjack-p1-c0', 'catan-p8-c0', 'catan-p9-c3', 'poker-p2-c1', 'poker-p7-c0', 'texas_holdem-p1-c2', 'texas_holdem-p2-c2']
In Blackjack, is an Ace always worth one point, as it is by default in Standard Rummy? → ['blackjack', 'rummy']
    ['blackjack-p1-c0', 'blackjack-p2-c0', 'rummy-p2-c1', 'rummy-p5-c1']
In Blackjack, at what hand total may the dealer stop taking cards? → ['blackjack']
    ['blackjack-p1-c0', 'blackjack-p2-c0', 'blackjack-p2-c1', 'blackjack-p1-c1']


In [17]:
for r in results:
    if r["question_type"] == "multi_hop":
        print(r["id"], r["verdict"], detect_games(r["question"]), r["chunks"])

2 FAIL [] catan-p8-c2;catan-p9-c0;catan-p8-c0;catan-p3-c1;catan-p12-c0;monopoly-p1-c1;monopoly-p2-c5;rummy-p6-c1;rummy-p3-c1
4 PASS ['catan'] catan-p6-c2;catan-p9-c0;catan-p9-c1;catan-p6-c0;catan-p3-c1
8 FAIL [] monopoly-p2-c3;monopoly-p1-c2;monopoly-p1-c1;monopoly-p2-c5;monopoly-p1-c4
14 FAIL ['rummy'] rummy-p3-c1;rummy-p5-c1;rummy-p4-c1;rummy-p6-c0;rummy-p1-c2
21 FAIL [] texas_holdem-p1-c2;blackjack-p1-c1;poker-p2-c0;blackjack-p2-c1;rummy-p2-c2
27 PASS [] texas_holdem-p3-c0;poker-p3-c3;poker-p3-c0;texas_holdem-p3-c1;poker-p3-c1;blackjack-p2-c0;blackjack-p1-c1;catan-p12-c0;catan-p8-c3;rummy-p7-c0;rummy-p6-c0
33 PASS [] texas_holdem-p3-c2;texas_holdem-p3-c3;texas_holdem-p4-c0;texas_holdem-p3-c1;poker-p3-c0;blackjack-p1-c1;blackjack-p1-c0;catan-p12-c0;catan-p10-c0;monopoly-p2-c1;monopoly-p2-c2;rummy-p3-c1;rummy-p7-c0


In [18]:
import inspect
print(inspect.getsource(retriever.search_routed))

    def search_routed(self, question, k=5, per_game=2, floor=-10.0):
        named = detect_games(question)

        # 1. one game named: search only that game
        if len(named) == 1:
            chunks, _ = self.search_in_game(question, named[0], k)
            return chunks

        # 2. no game named: global top k, plus any missing games above the floor
        if not named:
            results = self.search(question, k=k)
            have = {c["game"] for c in results}
            for game in sorted({c["game"] for c in self.chunks}):
                if game in have:
                    continue
                chunks, best = self.search_in_game(question, game, per_game)
                if best > floor:
                    results.extend(chunks)
            return results

        # 3. two or more games named: a few chunks from each named game
        results = []
        for game in named:
            chunks, _ = self.search_in_game(question, game, per_game)
            results

In [19]:
for r in results:
    if r["question_type"] == "multi_hop":
        n = len(r["chunks"].split(";"))
        games = sorted({c.split("-p")[0] for c in r["chunks"].split(";")})
        print(r["id"], r["verdict"], n, "chunks", games)

2 FAIL 9 chunks ['catan', 'monopoly', 'rummy']
4 PASS 5 chunks ['catan']
8 FAIL 5 chunks ['monopoly']
14 FAIL 5 chunks ['rummy']
21 FAIL 5 chunks ['blackjack', 'poker', 'rummy', 'texas_holdem']
27 PASS 11 chunks ['blackjack', 'catan', 'poker', 'rummy', 'texas_holdem']
33 PASS 13 chunks ['blackjack', 'catan', 'monopoly', 'poker', 'rummy', 'texas_holdem']
